# Multimodal Cancer Classification Challenge 2026 — v20 (paper-aligned single-model push)

**Goal: top the LB without ensembling.** v20 is now aligned with **Lian/Lindblad et al. 2025 "Let it shine"** — the same team's published recipe on this exact dataset that beats human performance (F1=83.34, Acc=91.79). Two MIL/no-mixup choices from v19 are reversed based on paper evidence.

## LB context (May 16, 2026)

| # | Team | Score |
|---|---|---|
| 1 | Group 1 (3 members) | **0.7832** |
| 2 | Group 2 | 0.7713 |
| **3** | **Rafael (you)** | **0.6479** (v11) |

To **top the LB** need ≥ **0.79**. To hit **teacher's 0.85 target** need +0.20 from current.

## Paper findings driving v20

**"Let it shine" (Lian, Lindblad et al. 2025, arXiv 2407.01869, Comp Bio Med 109498)** — same team, same dataset:

- **Backbone**: ResNet-50 ImageNet pretrained ✅ (we already do this)
- **Input**: 256→224 resize ✅ (we already do this)
- **Loss**: **SIL** (per-cell BCE) — explicitly rejects MIL per their 2024 PLOS One paper ❌ (we had MIL_WEIGHT=0.5; v20 sets to 0)
- **Mixup α=0.8** as primary regularizer ❌ (we had MIXUP_ALPHA=0; v20 sets to 0.8)
- AdamW LR 8e-5, weight decay 0.1, cosine annealing, 30 epochs, no TTA
- Late fusion (concat) beat by intermediate fusion (CAFNet) — kept for v21

## What changed vs v19 → v20

| Lever | v19 | **v20** | Source |
|---|---|---|---|
| Backbone | EffNet-B0 (~11M dual) | **ResNet-50 (~52M dual)** | L1a slide 30 + "Let it shine" |
| Input resolution | 128 native | **224 upscaled** | code-review CRITICAL + paper recipe |
| Validation | none | **2 patients held out, best-val ckpt** | L4 "Always do this" |
| LR strategy | single 3e-4 | **Discriminative: backbone 3e-5, head 3e-4** | L4 slide 96 |
| **MIL aux loss** | weight 0.5 | **OFF** | **Lindblad MIL-vs-SIL paper: SIL > MIL** |
| **Mixup** | OFF | **ON, α=0.8** | **"Let it shine" primary regularizer** |
| Dropout | 0.3 | **0.5** | code-review (larger model = more reg needed) |
| Label smoothing | 0 | **0.05** | L4 slide 48 |
| Multi-scale TTA | OFF | **ON, scales (192, 224, 256)** | L4 slide 79 |
| Ensemble | none | **none (still single — user constraint)** | "no ensemble until last" |

## Why these two paper-driven flips matter most

**1. MIL OFF** — v19's training trace showed MIL loss collapsed to 0.019 (model perfectly memorized per-patient signatures). Lindblad's MIL-vs-SIL paper explicitly concludes SIL outperforms ABMIL for OC detection. Our aux MIL was likely pulling the model toward train-patient artifacts and hurting OOD.

**2. Mixup α=0.8 ON** — heavy mixup is the paper's primary regularizer. v15 used α=0.1 (very light), v17+ disabled it. Restoring at the paper's α=0.8 gives strong implicit ensembling and improves OOD by training on convex combinations.

These are the two changes I expect to move LB the most for v20.

## What v20 still does NOT match the paper

| Setting | Paper | v20 | Why kept |
|---|---|---|---|
| LR | 8e-5 | 3e-4 head / 3e-5 backbone | Disc LR is L4-cited; paper's 8e-5 is single LR. Roughly similar effective backbone LR. |
| Weight decay | 0.1 | 1e-4 | Paper's 0.1 is very heavy; risk of under-fitting in our 12-epoch budget |
| Epochs | 30 | 12 | Compute budget — 30 would be ~7h training alone |
| Scheduler | CosineAnnealing | OneCycleLR | Both work fine for 12 epochs |
| TTA | None | 24-way multi-scale | Extra single-model lift, no downside |
| Fusion | CAFNet intermediate | Late (concat) | CAFNet code change is a v21-sized rewrite |
| Per-modality aug | Posterize/blur/solarize | ColorJitter + RandomErasing + affine | Our v19 aug is different but additive |

These are v21 candidates if v20 falls short of 0.78.

## Compute budget on T4

| Stage | Time |
|---|---|
| JPEG cache (one-time) | ~70 min |
| Test pixel stats | ~30 s |
| Train ResNet-50 @ 224 (12 ep × ~12 min/ep) + val pass | ~150 min |
| 24-way multi-scale TTA + AdaBN @ 224 | ~40 min |
| **Total** | **~4h 20min** |

## Reading the v20 LB

- **LB ≥ 0.78**: Tied with #1 leader. v21 = CAFNet intermediate fusion to push toward 0.85.
- **LB 0.70–0.78**: Strong progress, in striking distance. v21 = either CAFNet or 3-fold CV ensemble.
- **LB 0.65–0.70**: Recipe is correct but missing something. v21 swap to DenseNet-201 or extend to 30 epochs.
- **LB < 0.65**: Val curve diagnoses — train↔val gap → over-regularization (drop mixup or dropout); both low → under-training (more epochs).

## IMPORTANT — DO NOT JUST CLICK RUN ALL

To get a `submission.csv` you must:

1. Click **Save Version** (top-right green button)
2. Choose **Save & Run All (Commit)**
3. Description: `"v20: ResNet-50 @ 224 + MIL OFF + mixup 0.8 + val + disc LR (paper-aligned)"`
4. **Wait ~4h 20min**
5. Open the saved version → Output tab → submission.csv is there

## Sources

- Lian, Lindblad, Runow Stark, Hirsch, Sladoje. *"Let it shine: Autofluorescence of Papanicolaou-stain improves AI-based cytological oral cancer detection"*. Comp Bio Med 2025. arXiv:2407.01869.
- Koriakina, Sladoje, Lindblad et al. *"Deep multiple instance learning versus conventional deep single instance learning for interpretable oral cancer detection"*. PLOS One 2024. (https://github.com/MIDA-group/OralCancerMILvsSIL)
- L1a Intro lecture, slide 30 (Lindblad's oral cancer paper recap)
- L4 Learning Process lecture, slides 48 (label smoothing), 79 (TTA), 96 (discriminative LR)

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/rafaelproena/a3-adl"),
    Path("/kaggle/input/a3-adl"),
    Path("/kaggle/input/competitions/multimodal-cancer-classification-challenge-2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if (p / "train.csv").exists()), None)
assert DATA_ROOT is not None, f"train.csv not found at any of {DATA_ROOT_CANDIDATES}"
print("DATA_ROOT =", DATA_ROOT)

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === v20: backbone choice (L1a slide 30 + Lian 2025) ===
USE_RESNET50        = True
USE_EFFICIENTNET    = False

# === v20: input resolution — matches Lian 2025 (256→224) and ResNet-50 pretrain ===
INPUT_SIZE          = 224

# === v20: validation tracking — L4 ===
USE_VAL             = True
VAL_PATIENTS        = [16, 11]

# === v20: discriminative LR (code-review HIGH issue #3 — head LR tuned closer to paper) ===
USE_DISCRIMINATIVE_LR = True
# Paper uses single LR 8e-5. We split as discriminative for L4 alignment, but pulling head LR
# down from 3e-4 → 1e-4 to keep it closer to paper's effective head LR. Backbone stays 1e-5 (= LR_HEAD / 10).
LR_HEAD             = 1e-4   # CHANGED from 3e-4 per code-review issue #3
LR_BACKBONE_RATIO   = 0.1    # backbone_lr = 1e-5

# === v20: LOSS configuration (Koriakina 2024 PLOS One: SIL > MIL) ===
USE_MIL_LOSS        = False
MIL_WEIGHT          = 0.0

# === v20: MIXUP α=0.8 (Lian 2025 primary regularizer) ===
MIXUP_ALPHA         = 0.8

# === v20: LABEL SMOOTHING — code-review CRITICAL #1: disabled under heavy mixup ===
# Mixup already provides implicit label smoothing via convex combinations. Stacking
# explicit eps=0.05 on top is double-regularization that may slow convergence at 12 ep.
# Lian 2025 uses mixup WITHOUT explicit label smoothing.
LABEL_SMOOTHING     = 0.0    # CHANGED from 0.05 per code-review (paper uses mixup alone)

# === v19 augmentation kept ===
USE_STRONG_AUG      = True
RANDOM_ERASING_P    = 0.25

# === Kept from v17 (always ON) ===
USE_TEST_STAIN_NORM = True
USE_ADABN           = True
USE_MULTISCALE_TTA  = True
TTA_SCALES          = (192, 224, 256)

# === Training ===
BASE_SEED   = 1
EPOCHS      = 12
BATCH_SIZE  = 64
GRAD_ACCUM  = 2              # effective batch 128
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
DROPOUT     = 0.5
LR          = LR_HEAD        # backward-compat alias

NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# v11 hardcoded stats (used when USE_TEST_STAIN_NORM=False)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(BASE_SEED)

# Sanity: only one backbone selected
assert not (USE_RESNET50 and USE_EFFICIENTNET), "Pick at most one of RESNET50 / EFFICIENTNET"
_backbone_name = ("ResNet-50" if USE_RESNET50
                  else "EfficientNet-B0" if USE_EFFICIENTNET
                  else "ResNet-18")

print(f"\nConfig (v20 — paper-aligned + code-review tuning):")
print(f"  Backbone            = {_backbone_name}")
print(f"  INPUT_SIZE          = {INPUT_SIZE} (matches ImageNet pretrain + paper recipe)")
print(f"  BATCH_SIZE          = {BATCH_SIZE} * GRAD_ACCUM={GRAD_ACCUM} (effective {BATCH_SIZE*GRAD_ACCUM})")
print(f"  DROPOUT             = {DROPOUT}")
print(f"  LABEL_SMOOTHING     = {LABEL_SMOOTHING}  (off — mixup provides implicit smoothing)")
print(f"  MIXUP_ALPHA         = {MIXUP_ALPHA}  (paper recipe; was 0.0 in v19)")
print(f"  USE_MIL_LOSS        = {USE_MIL_LOSS}  weight={MIL_WEIGHT}  (paper uses SIL not MIL)")
print(f"  USE_VAL             = {USE_VAL}  val_patients={VAL_PATIENTS}")
print(f"  USE_DISCRIMINATIVE_LR = {USE_DISCRIMINATIVE_LR}  head_lr={LR_HEAD}  "
      f"backbone_lr={LR_HEAD * LR_BACKBONE_RATIO if USE_DISCRIMINATIVE_LR else LR_HEAD}")
print(f"  USE_STRONG_AUG      = {USE_STRONG_AUG}  erasing_p={RANDOM_ERASING_P}")
print(f"  USE_TEST_STAIN_NORM = {USE_TEST_STAIN_NORM}")
print(f"  USE_ADABN           = {USE_ADABN}")
print(f"  USE_MULTISCALE_TTA  = {USE_MULTISCALE_TTA}  scales={TTA_SCALES}")
print(f"  EPOCHS              = {EPOCHS}")

In [ ]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """v19: also returns patient_id for MIL grouping in training."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
        # Patient ID is -1 for test rows (Name has no pat_X prefix in some splits but in this
        # dataset all names are pat_NN_image_MM.jpg so parse always succeeds).
        self.has_pid = "patient_id" in self.df.columns
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        pid = int(row["patient_id"]) if self.has_pid else -1
        return {"bf": bf, "fl": fl, "label": label, "patient_id": pid, "name": name}

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

In [ ]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd

def _make_resnet50_branch(pretrained=True):
    """v20 NEW: ResNet-50 dual-branch builder per L1a slide 30 recommendation."""
    weights = "DEFAULT" if pretrained else None
    net = models.resnet50(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features  # 2048 for ResNet-50
    net.fc = nn.Identity()
    return net, fd

def _make_effnet_b0_branch(pretrained=True):
    import timm
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                            num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                        stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

def _make_branch(pretrained=True):
    if USE_RESNET50:
        return _make_resnet50_branch(pretrained)
    if USE_EFFICIENTNET:
        return _make_effnet_b0_branch(pretrained)
    return _make_resnet18_branch(pretrained)

class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_branch(pretrained)
        self.fl_branch, _  = _make_branch(pretrained)
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        self._feat_dim = fd

    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

    def param_groups(self, head_lr, backbone_ratio=0.1, weight_decay=1e-4):
        """v20 NEW: build optimizer param groups for discriminative LR.
        Backbone params get head_lr * backbone_ratio (default 1:10 per L4 slide 96).
        Head params get head_lr.
        """
        backbone_lr = head_lr * backbone_ratio
        backbone_params = (list(self.bf_branch.parameters()) +
                           list(self.fl_branch.parameters()))
        head_params = list(self.head.parameters())
        return [
            {"params": backbone_params, "lr": backbone_lr, "weight_decay": weight_decay,
             "name": "backbone"},
            {"params": head_params,     "lr": head_lr,     "weight_decay": weight_decay,
             "name": "head"},
        ]

with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    n_params = sum(p.numel() for p in _m.parameters())
    n_backbone = (sum(p.numel() for p in _m.bf_branch.parameters()) +
                  sum(p.numel() for p in _m.fl_branch.parameters()))
    n_head = sum(p.numel() for p in _m.head.parameters())
    print(f"Output shape: {_m(_x, _x).shape}")
    print(f"Params:       {n_params/1e6:.1f}M total = {n_backbone/1e6:.1f}M backbone + {n_head/1e6:.1f}M head")
    print(f"Backbone:     {_backbone_name}  (feature dim {_m._feat_dim})")
    if USE_DISCRIMINATIVE_LR:
        pg = _m.param_groups(LR_HEAD, LR_BACKBONE_RATIO, WEIGHT_DECAY)
        print(f"Param groups: {len(pg)} groups | "
              f"backbone_lr={pg[0]['lr']}  head_lr={pg[1]['lr']}")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

# Compute pixel statistics for stain normalization.
def _sample_pixel_stats(cache, names, n_sample=1500):
    rng = np.random.default_rng(42)
    sampled = rng.choice(names, size=min(n_sample, len(names)), replace=False)
    pixels = []
    for n in sampled:
        img = np.asarray(Image.open(io.BytesIO(cache[n])).convert("L"),
                         dtype=np.float32) / 255.0
        pixels.append(img.ravel())
    pixels = np.concatenate(pixels)
    return float(pixels.mean()), float(pixels.std())

if USE_TEST_STAIN_NORM:
    BF_MEAN_T, BF_STD_T = _sample_pixel_stats(bf_test_cache, df_test["Name"].tolist())
    FL_MEAN_T, FL_STD_T = _sample_pixel_stats(fl_test_cache, df_test["Name"].tolist())
    BF_MEAN_R, BF_STD_R = _sample_pixel_stats(bf_train_cache, df_train["Name"].tolist())
    FL_MEAN_R, FL_STD_R = _sample_pixel_stats(fl_train_cache, df_train["Name"].tolist())
    print(f"\nPixel statistics:")
    print(f"  BF train: mean={BF_MEAN_R:.4f} std={BF_STD_R:.4f}")
    print(f"  BF test:  mean={BF_MEAN_T:.4f} std={BF_STD_T:.4f}")
    print(f"  FL train: mean={FL_MEAN_R:.4f} std={FL_STD_R:.4f}")
    print(f"  FL test:  mean={FL_MEAN_T:.4f} std={FL_STD_T:.4f}")
    BF_MEAN, BF_STD = BF_MEAN_T, BF_STD_T
    FL_MEAN, FL_STD = FL_MEAN_T, FL_STD_T
    print(f"  -> using TEST stats (USE_TEST_STAIN_NORM=True)")
else:
    print(f"\nUsing v11 hardcoded normalization (USE_TEST_STAIN_NORM=False)")

In [ ]:
def _to_tensor_norm(mean, std):
    """v20: ToTensor -> Resize(INPUT_SIZE) -> Normalize.
    Resize to INPUT_SIZE (default 224) so the network sees the resolution its ImageNet
    pretrained weights were trained at. Bilinear upscale from 128 is cheap.
    """
    def fn(img):
        t = TF.to_tensor(img)  # [1, 128, 128]
        if INPUT_SIZE != t.shape[-1]:
            t = TF.resize(t, [INPUT_SIZE, INPUT_SIZE], antialias=True)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    """D4 + small rotation, applied identically to BF and FL (paired)."""
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0,
                 affine_deg=0.0, affine_translate=0.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
        self.affine_deg = affine_deg; self.affine_translate = affine_translate
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        # v19: paired affine — same translate applied to both modalities (keeps BF/FL aligned)
        if self.affine_deg > 0 or self.affine_translate > 0:
            H, W = bf.shape[-2], bf.shape[-1]
            angle = random.uniform(-self.affine_deg, self.affine_deg) if self.affine_deg > 0 else 0.0
            tx = random.uniform(-self.affine_translate, self.affine_translate) * W if self.affine_translate > 0 else 0
            ty = random.uniform(-self.affine_translate, self.affine_translate) * H if self.affine_translate > 0 else 0
            bf = TF.affine(bf, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
            fl = TF.affine(fl, angle=angle, translate=(int(tx), int(ty)), scale=1.0, shear=0.0)
        return bf, fl

def train_modality_transform(modality):
    """v19: stronger color jitter, with optional RandomErasing applied after normalization."""
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    if USE_STRONG_AUG:
        steps = [T.ColorJitter(brightness=0.4, contrast=0.4), norm]
        if RANDOM_ERASING_P > 0:
            steps.append(T.RandomErasing(p=RANDOM_ERASING_P, scale=(0.02, 0.20),
                                         ratio=(0.3, 3.3), value=0.0))
        return T.Compose(steps)
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

def build_paired_aug():
    if USE_STRONG_AUG:
        return PairedGeoAug(max_rot=10.0, affine_deg=15.0, affine_translate=0.10)
    return PairedGeoAug(max_rot=10.0)

print(f"Augmentation summary:")
print(f"  Input resize:     128 -> {INPUT_SIZE}  (matches ImageNet pretrain)")
print(f"  ColorJitter:      {'brightness/contrast=0.4' if USE_STRONG_AUG else 'brightness/contrast=0.2'}")
print(f"  RandomErasing:    p={RANDOM_ERASING_P if USE_STRONG_AUG else 0.0}")
print(f"  PairedGeoAug:     D4 + ±10° rot" + (" + ±15° affine + 10% translate" if USE_STRONG_AUG else ""))

In [ ]:
def smooth(y, eps):
    if eps <= 0: return y
    return y * (1.0 - eps) + eps * 0.5

def mixup_batch(bf, fl, y, alpha=0.8):
    """v20 paper-aligned: heavy mixup (alpha=0.8 from 'Let it shine')."""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(bf.size(0), device=bf.device)
    bf_mix = lam * bf + (1 - lam) * bf[idx]
    fl_mix = lam * fl + (1 - lam) * fl[idx]
    return bf_mix, fl_mix, y, y[idx], lam

def mil_patient_loss(logits, y, patient_ids, pos_weight=None):
    """Kept for ablation but disabled by default in v20 (SIL > MIL per Lindblad's MIL-vs-SIL paper)."""
    unique_pids = torch.unique(patient_ids)
    if len(unique_pids) < 2:
        return torch.zeros((), device=logits.device, dtype=logits.dtype)
    p_logits, p_labels = [], []
    for pid in unique_pids:
        mask = patient_ids == pid
        p_logits.append(logits[mask].mean())
        p_labels.append(y[mask][0])
    p_logits = torch.stack(p_logits)
    p_labels = torch.stack(p_labels)
    return F.binary_cross_entropy_with_logits(p_logits, p_labels.float(),
                                              pos_weight=pos_weight)

def run_epoch_train(model, loader, optimizer, scaler, criterion_cell, sched,
                    pos_weight=None, grad_accum=1, log_every=200):
    """v20: mixup support added back (was disabled in v19 due to MIL conflict)."""
    model.train()
    losses, hard_ys, ps = [], [], []
    cell_losses, mil_losses = [], []
    t_last = time.time()
    optimizer.zero_grad(set_to_none=True)
    accum_count = 0
    for i, batch in enumerate(loader):
        bf  = batch["bf"].to(DEVICE, non_blocking=True)
        fl  = batch["fl"].to(DEVICE, non_blocking=True)
        y   = batch["label"].float().to(DEVICE, non_blocking=True)
        pid = batch["patient_id"].to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())

        # v20: paper-aligned heavy mixup (alpha=0.8) — only when MIL is off
        if MIXUP_ALPHA > 0 and not USE_MIL_LOSS:
            bf_in, fl_in, y_a, y_b, lam = mixup_batch(bf, fl, y, MIXUP_ALPHA)
            y_a_s = smooth(y_a, LABEL_SMOOTHING)
            y_b_s = smooth(y_b, LABEL_SMOOTHING)
        else:
            bf_in, fl_in = bf, fl
            y_a_s = smooth(y, LABEL_SMOOTHING)
            y_b_s = None
            lam = 1.0

        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf_in, fl_in)
            if y_b_s is not None:
                loss_cell = lam * criterion_cell(logits, y_a_s) + (1.0 - lam) * criterion_cell(logits, y_b_s)
            else:
                loss_cell = criterion_cell(logits, y_a_s)
            if USE_MIL_LOSS:
                loss_mil = mil_patient_loss(logits, y, pid, pos_weight=pos_weight)
                loss = loss_cell + MIL_WEIGHT * loss_mil
            else:
                loss_mil = torch.zeros((), device=logits.device)
                loss = loss_cell
            loss_scaled = loss / grad_accum

        if scaler is not None:
            scaler.scale(loss_scaled).backward()
        else:
            loss_scaled.backward()
        accum_count += 1
        if accum_count >= grad_accum:
            if scaler is not None:
                if GRAD_CLIP > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale: sched.step()
            else:
                if GRAD_CLIP > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step(); sched.step()
            optimizer.zero_grad(set_to_none=True)
            accum_count = 0

        losses.append(loss.item())
        cell_losses.append(loss_cell.item())
        mil_losses.append(loss_mil.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | "
                  f"total {float(np.mean(losses[-log_every:])):.4f} "
                  f"cell {float(np.mean(cell_losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    # Train AUC is noisy under mixup (preds use mixed inputs, labels are original) — still useful as a trend signal
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), float(np.mean(cell_losses)), float(np.mean(mil_losses)), auc

@torch.no_grad()
def run_epoch_val(model, loader):
    """v20: forward-only pass over val set, returns cell-AUC and patient-AUC."""
    model.eval()
    ys, ps, pids = [], [], []
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(bf, fl)
        ps.append(torch.sigmoid(logits).float().cpu().numpy())
        ys.append(batch["label"].numpy())
        pids.append(batch["patient_id"].numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps); pids = np.concatenate(pids)
    cell_auc = roc_auc_score(ys, ps) if len(np.unique(ys)) > 1 else float("nan")
    df_p = pd.DataFrame({"y": ys, "p": ps, "pid": pids}).groupby("pid").agg(
        y=("y", "first"), p=("p", "mean")).reset_index()
    patient_auc = (roc_auc_score(df_p["y"], df_p["p"])
                   if len(df_p["y"].unique()) > 1 else float("nan"))
    return cell_auc, patient_auc

# === v20: split off the val patients ===
if USE_VAL:
    val_mask = df_train["patient_id"].isin(VAL_PATIENTS)
    df_train_use = df_train[~val_mask].reset_index(drop=True)
    df_val       = df_train[ val_mask].reset_index(drop=True)
    print(f"\nVal split:")
    print(f"  val patients = {sorted(df_val['patient_id'].unique().tolist())}  "
          f"({len(df_val)} cells, pos_rate {df_val['Diagnosis'].mean():.3f})")
    print(f"  train patients = {sorted(df_train_use['patient_id'].unique().tolist())}  "
          f"({len(df_train_use)} cells, pos_rate {df_train_use['Diagnosis'].mean():.3f})")
else:
    df_train_use = df_train
    df_val = None

print(f"\n=== v20: Training {_backbone_name} @ {INPUT_SIZE}x{INPUT_SIZE} "
      f"({EPOCHS} epochs, {df_train_use['patient_id'].nunique()} train patients"
      f"{', val tracked' if USE_VAL else ', no val'}) ===")
seed_everything(BASE_SEED + 100)
train_ds = CachedCellDataset(df_train_use, bf_train_cache, fl_train_cache,
                             train_modality_transform("bf"),
                             train_modality_transform("fl"),
                             paired_tf=build_paired_aug())
sampler = PatientBalancedSampler(df_train_use, batch_size=BATCH_SIZE,
                                 patients_per_batch=PATIENTS_PER_BATCH, seed=BASE_SEED + 100)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

if USE_VAL:
    val_ds = CachedCellDataset(df_val, bf_train_cache, fl_train_cache,
                               eval_modality_transform("bf"),
                               eval_modality_transform("fl"))
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
pos = (df_train_use["Diagnosis"] == 1).sum()
neg = (df_train_use["Diagnosis"] == 0).sum()
pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
criterion_cell = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# v20: discriminative LR via param groups
if USE_DISCRIMINATIVE_LR:
    optimizer = torch.optim.AdamW(
        model.param_groups(head_lr=LR_HEAD, backbone_ratio=LR_BACKBONE_RATIO,
                           weight_decay=WEIGHT_DECAY))
    print(f"  Optimizer: AdamW, discriminative LR "
          f"(backbone {LR_HEAD*LR_BACKBONE_RATIO}, head {LR_HEAD})")
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    print(f"  Optimizer: AdamW, single LR {LR_HEAD}")
print(f"  pos_weight={pos_weight.item():.3f}  "
      f"MIXUP_ALPHA={MIXUP_ALPHA}  USE_MIL_LOSS={USE_MIL_LOSS}  "
      f"label_smoothing={LABEL_SMOOTHING}  dropout={DROPOUT}")

# OneCycleLR — adjust step count for grad accumulation
group_max_lrs = [g["lr"] for g in optimizer.param_groups]
steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=group_max_lrs, steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS, pct_start=0.1)
scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None
print(f"  steps/epoch={len(train_loader)} (raw) / {steps_per_epoch} (after grad_accum={GRAD_ACCUM})")

history = []
best_val_auc = -1.0
best_epoch = -1
last_ckpt_path  = OUT_DIR / "last.pt"
best_ckpt_path  = OUT_DIR / "best_val.pt"

for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_cell, tr_mil, tr_auc = run_epoch_train(
        model, train_loader, optimizer, scaler, criterion_cell, sched,
        pos_weight=pos_weight, grad_accum=GRAD_ACCUM)
    val_cell_auc, val_patient_auc = (
        run_epoch_val(model, val_loader) if USE_VAL else (float("nan"), float("nan"))
    )
    dt = time.time() - t0
    msg = (f"  ep {ep:>2d} | total {tr_loss:.4f} cell {tr_cell:.4f} mil {tr_mil:.4f} "
           f"tr_auc {tr_auc:.4f}")
    if USE_VAL:
        msg += f" | val_cell {val_cell_auc:.4f} val_pat {val_patient_auc:.4f}"
    msg += f" | {dt:.1f}s"
    print(msg)
    history.append({
        "epoch": ep, "tr_loss": tr_loss, "tr_cell": tr_cell, "tr_mil": tr_mil,
        "tr_auc": tr_auc, "val_cell_auc": val_cell_auc, "val_patient_auc": val_patient_auc,
        "time": dt,
    })
    # v20: save best-val-AUC ckpt
    if USE_VAL and val_cell_auc > best_val_auc:
        best_val_auc = val_cell_auc
        best_epoch = ep
        torch.save({"model": model.state_dict(), "epoch": ep,
                    "args": {"dropout": DROPOUT, "backbone": _backbone_name,
                             "input_size": INPUT_SIZE,
                             "val_cell_auc": val_cell_auc,
                             "val_patient_auc": val_patient_auc}},
                   best_ckpt_path)
    torch.save({"model": model.state_dict(), "epoch": ep,
                "args": {"dropout": DROPOUT, "backbone": _backbone_name,
                         "input_size": INPUT_SIZE}},
               last_ckpt_path)

with open(OUT_DIR / "history.json", "w") as f:
    json.dump({"history": history,
               "best_epoch": best_epoch,
               "best_val_auc": best_val_auc}, f, indent=2)

# Pick ckpt for inference
if USE_VAL and best_epoch >= 0:
    ckpt_path = best_ckpt_path
    print(f"\nUsing best-val ckpt: epoch {best_epoch}, val_cell_auc {best_val_auc:.4f}")
else:
    ckpt_path = last_ckpt_path
    print(f"\nUsing last-epoch ckpt: epoch {EPOCHS - 1}")

del model, optimizer, sched, scaler, train_loader
if USE_VAL: del val_loader
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
epochs = [e["epoch"] for e in history]
# Panel 1: losses
ax[0].plot(epochs, [e["tr_loss"] for e in history], marker="o", color="tab:blue", label="total")
ax[0].plot(epochs, [e["tr_cell"] for e in history], marker="s", color="tab:purple", label="cell BCE")
ax[0].plot(epochs, [e["tr_mil"]  for e in history], marker="^", color="tab:orange", label="MIL patient BCE")
ax[0].legend(); ax[0].set(title="Train losses", xlabel="epoch", ylabel="loss")
# Panel 2: train vs val AUC (the diagnostic panel)
ax[1].plot(epochs, [e["tr_auc"] for e in history], marker="o", color="tab:green", label="train (cell)")
if USE_VAL:
    ax[1].plot(epochs, [e["val_cell_auc"]    for e in history], marker="s", color="tab:red",  label="val (cell)")
    ax[1].plot(epochs, [e["val_patient_auc"] for e in history], marker="^", color="tab:brown", label="val (patient)")
    # Mark best-val-AUC epoch with vertical line
    if best_epoch >= 0:
        ax[1].axvline(best_epoch, color="grey", linestyle="--", alpha=0.5,
                      label=f"best @ ep {best_epoch}")
ax[1].legend(); ax[1].set(title="AUC: train vs val", xlabel="epoch", ylabel="AUC")
ax[1].set_ylim(0.4, 1.02)
# Panel 3: epoch time
ax[2].plot(epochs, [e["time"] for e in history], marker="o", color="tab:red")
ax[2].set(title="Epoch time (s)", xlabel="epoch", ylabel="seconds")
for a in ax: a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# === AdaBN: update BN running stats on test data before inference ===
@torch.no_grad()
def adabn_pass(model, loader):
    model.train()
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            _ = model(bf, fl)
    model.eval()

def _d4_at_scale(bf, fl, scale=None):
    if scale is not None and scale != bf.shape[-1]:
        bf = F.interpolate(bf, size=(scale, scale), mode="bilinear", align_corners=False)
        fl = F.interpolate(fl, size=(scale, scale), mode="bilinear", align_corners=False)
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def _multiscale_tta(bf, fl, scales):
    for s in scales:
        for bf_t, fl_t in _d4_at_scale(bf, fl, scale=s):
            yield bf_t, fl_t

def load_model(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict(ckpt_path, loader, tta_scales=None):
    model = load_model(ckpt_path)
    if USE_ADABN:
        print("  running AdaBN pass...")
        adabn_pass(model, loader)
    aug_fn = (lambda bf, fl: _multiscale_tta(bf, fl, tta_scales)) if tta_scales \
             else (lambda bf, fl: _d4_at_scale(bf, fl))
    n_aug = 8 * (len(tta_scales) if tta_scales else 1)
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in aug_fn(bf, fl):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

# v20: at INPUT_SIZE=224 with ResNet-50, drop inference batch from 256 to 96 to stay
# within T4 16GB VRAM. (256 fit at 128x128 but pushes limits at 224x224 + multi-scale 256.)
INFER_BATCH = 96 if INPUT_SIZE >= 192 else 256
test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=INFER_BATCH, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

scales_to_use = TTA_SCALES if USE_MULTISCALE_TTA else None
n_aug_total = 8 * (len(TTA_SCALES) if USE_MULTISCALE_TTA else 1)
print(f"Predicting with {n_aug_total}-way TTA "
      f"(scales={scales_to_use or 'native'}, AdaBN={USE_ADABN}, batch={INFER_BATCH})")

t0 = time.time()
preds = predict(ckpt_path, test_loader, tta_scales=scales_to_use)
print(f"  done in {time.time()-t0:.1f}s")

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")
print(sub.head())
!wc -l /kaggle/working/submission.csv